## NSR for KINSHIP with Delta Layer (Symmetry + Invertibility)


- 原有 `alpha-beta-gamma` 三层结构
- 新增高阶抽象层 `delta`（仅实现 `Symmetry` 与 `Invertibility`）
- 训练时加入 `W_{beta,gamma}` 的 Hebbian 增强
- 推理时按优先级：`Symmetry > Invertibility > Base RR`


## Full NSR On $\texttt{KINSHIP 1990}$

### Model Definition
- 注意，这就是Full NSR，不过，这是一个没法完成汇聚拓扑和发散拓扑的版本
- 下面的Enhanced NSR版本才其实才是真正的“Full NSR”

In [1]:
# ===================== Full NSR On KINSHIP 1990（独立主实验）=====================

import re
import time
import random
from pathlib import Path
from dataclasses import dataclass
from collections import defaultdict

import numpy as np
import pandas as pd

print("=" * 100)
print("Full NSR On KINSHIP 1990")
print("- strict / relaxed compositionality")
print("- parallel invertibility + compositionality")
print("- vectorized scoring + sparse compositionality")
print("=" * 100)


@dataclass
class RuleLearnConfig:
    symmetry_threshold: float = 0.5
    inverse_threshold: float = 0.5
    min_support: int = 8
    min_conf: float = 0.6
    composition_mode: str = "relaxed"



def load_kinship_splits(path: Path, split_from_single_file=False, train_r=0.7, val_r=0.1, seed=42):
    entity_to_id = {}
    relation_to_id = {}

    def get_entity_id(name):
        if name not in entity_to_id:
            entity_to_id[name] = len(entity_to_id)
        return entity_to_id[name]

    def get_relation_id(name):
        if name not in relation_to_id:
            relation_to_id[name] = len(relation_to_id)
        return relation_to_id[name]

    pattern = re.compile(r"([^\(]+)\(([^,]+),\s*([^\)]+)\)")
    
    def parse_file(p: Path):
        triples = []
        if not p.exists():
            return triples
        for raw_line in p.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line:
                continue
            match = pattern.match(line)
            if not match:
                continue
            rel = match.group(1).strip()
            head = match.group(2).strip()
            tail = match.group(3).strip()
            h = get_entity_id(head)
            r = get_relation_id(rel)
            t = get_entity_id(tail)
            triples.append((h, r, t))
        return triples

    if split_from_single_file:
        random.seed(seed)
        triples = parse_file(path)
        
        # Group by relation
        by_rel = defaultdict(list)
        for h, r, t in triples:
            by_rel[r].append((h, r, t))
            
        train_triples, valid_triples, test_triples = [], [], []
        for r, r_triples in by_rel.items():
            random.shuffle(r_triples)
            n = len(r_triples)
            tr_end = int(n * train_r)
            val_end = tr_end + int(n * val_r)
            
            train_triples.extend(r_triples[:tr_end])
            valid_triples.extend(r_triples[tr_end:val_end])
            test_triples.extend(r_triples[val_end:])
            
    else:
        # Traditional loading from directory
        train_triples = parse_file(path / "train.data")
        valid_triples = parse_file(path / "valid.data")
        test_triples = parse_file(path / "test.data")
        
    all_triples = train_triples + valid_triples + test_triples

    id_to_entity = {idx: name for name, idx in entity_to_id.items()}
    id_to_relation = {idx: name for name, idx in relation_to_id.items()}
    return train_triples, valid_triples, test_triples, all_triples, entity_to_id, relation_to_id, id_to_entity, id_to_relation



def discover_sym_inv(train_triples, num_relations):
    pairs_by_r = defaultdict(set)
    pair_to_relations = defaultdict(set)
    for h, r, t in train_triples:
        pairs_by_r[r].add((h, t))
        pair_to_relations[(h, t)].add(r)

    symmetric_relations = set()
    for r in range(num_relations):
        pairs = pairs_by_r[r]
        if pairs and any((t, h) in pairs for (h, t) in pairs):
            symmetric_relations.add(r)

    inverse_pairs = set()
    for r1 in range(num_relations):
        p1 = pairs_by_r[r1]
        if not p1:
            continue
        for r2 in range(num_relations):
            if r1 == r2:
                continue
            p2 = pairs_by_r[r2]
            if not p2:
                continue
            if any((t, h) in p2 for (h, t) in p1):
                inverse_pairs.add((r1, r2))

    return pair_to_relations, symmetric_relations, inverse_pairs



def discover_composition_strict(train_triples, num_relations):
    pairs_by_r = defaultdict(set)
    for h, r, t in train_triples:
        pairs_by_r[r].add((h, t))

    rules = {}
    for r_i in range(num_relations):
        for r_j in range(num_relations):
            chain_pairs = set()
            for h, x in pairs_by_r[r_i]:
                for x2, t in pairs_by_r[r_j]:
                    if x == x2:
                        chain_pairs.add((h, t))
            if not chain_pairs:
                continue
            for r_k in range(num_relations):
                if all((h, t) in pairs_by_r[r_k] for (h, t) in chain_pairs):
                    rules[(r_i, r_j)] = r_k
                    break
    return rules



def discover_composition_relaxed(train_triples, num_relations, min_support=8, min_conf=0.6):
    pairs_by_r = defaultdict(set)
    for h, r, t in train_triples:
        pairs_by_r[r].add((h, t))

    rules = {}
    stats = {}
    for r_i in range(num_relations):
        left = pairs_by_r[r_i]
        if not left:
            continue
        for r_j in range(num_relations):
            right = pairs_by_r[r_j]
            if not right:
                continue

            chain_pairs = set()
            for h, x in left:
                for x2, t in right:
                    if x == x2:
                        chain_pairs.add((h, t))

            support = len(chain_pairs)
            if support < min_support:
                continue

            best_rk = None
            best_hit = -1
            for r_k in range(num_relations):
                hit = len(chain_pairs & pairs_by_r[r_k])
                if hit > best_hit:
                    best_hit = hit
                    best_rk = r_k

            conf = best_hit / support if support else 0.0
            if conf >= min_conf:
                rules[(r_i, r_j)] = best_rk
                stats[(r_i, r_j)] = {"r_k": best_rk, "support": support, "hit": best_hit, "conf": conf}

    return rules, stats


class NSR1990Compositionality:
    def __init__(self, num_entities, num_relations):
        self.N = num_entities
        self.M = num_relations
        self.W_beta_beta = np.zeros((self.N * self.M, self.N * self.M), dtype=float)
        self.W_beta_gamma = np.zeros((self.N, self.M), dtype=float)
        self.W1_delta_gamma = np.zeros((self.M, self.M), dtype=float)
        self.W1_gamma_delta = np.zeros((self.M, self.M), dtype=float)
        self.W2_delta_gamma = np.zeros((self.M, self.M), dtype=float)
        self.W2_gamma_delta = np.zeros((self.M, self.M), dtype=float)
        self.T3_delta_gamma_gamma = np.zeros((self.M, self.M, self.M), dtype=float)
        self.W3_gamma_delta = np.zeros((self.M, self.M), dtype=float)
        self.W3_delta_gamma = np.zeros((self.M, self.M), dtype=float)
        self.symmetric_relations = set()
        self.inverse_pairs = set()
        self.inverse_map = {}
        self.composition_rules = {}
        self.composition_rule_stats = {}
        self.train_triples = []
        self._pairs_by_r = defaultdict(set)
        self._tails_by_hr = defaultdict(list)
        self._score_cache = {}
        self._base_cache = {}
        self._comp_rules_by_output = defaultdict(list)

    def _vec_index(self, entity_idx, relation_idx):
        return relation_idx * self.N + entity_idx

    def fit(self, train_triples):
        self.train_triples = list(train_triples)
        self._score_cache.clear()
        self._base_cache.clear()
        self._pairs_by_r = defaultdict(set)
        self._tails_by_hr = defaultdict(list)

        hr_freq = defaultdict(int)
        for h, r, t in train_triples:
            src = self._vec_index(h, r)
            dst = self._vec_index(t, r)
            self.W_beta_beta[dst, src] = 1.0
            self.W_beta_gamma[h, r] = 1.0
            hr_freq[(h, r)] += 1
            self._pairs_by_r[r].add((h, t))
            self._tails_by_hr[(h, r)].append(t)

        for (h, r), freq in hr_freq.items():
            self.W_beta_gamma[h, r] = 1.0 + np.log1p(freq)
        return self

    def set_rules(self, symmetric_relations, inverse_pairs, composition_rules):
        self.symmetric_relations = set(symmetric_relations)
        self.inverse_pairs = set(inverse_pairs)
        self.inverse_map = {}
        self._comp_rules_by_output = defaultdict(list)

        for r in self.symmetric_relations:
            self.W1_delta_gamma[r, r] = 1.0
            self.W1_gamma_delta[r, r] = 1.0

        for r1, r2 in self.inverse_pairs:
            self.inverse_map[r1] = r2
            self.W2_delta_gamma[r1, r2] = 1.0
            self.W2_delta_gamma[r2, r1] = 1.0
            self.W2_gamma_delta[r1, r2] = 1.0
            self.W2_gamma_delta[r2, r1] = 1.0

        self.composition_rules = dict(composition_rules)
        for (r_i, r_j), r_k in self.composition_rules.items():
            self.T3_delta_gamma_gamma[r_k, r_i, r_j] = 1.0
            self.W3_gamma_delta[r_k, r_k] = 1.0
            self.W3_delta_gamma[r_k, r_k] = 1.0
            self._comp_rules_by_output[r_k].append((r_i, r_j))

        return self

    def _score_candidates_via_relation(self, h_q, r_used):
        key = (h_q, r_used)
        if key in self._score_cache:
            return dict(self._score_cache[key])

        row_idx = self._vec_index(h_q, r_used)
        row = self.W_beta_beta[row_idx, r_used * self.N:(r_used + 1) * self.N]
        scores_arr = row * self.W_beta_gamma[:, r_used]
        scores = {int(e): float(s) for e, s in enumerate(scores_arr) if s > 0}
        self._score_cache[key] = scores
        return dict(scores)

    def _base_rr_infer(self, h_q, r_q):
        key = (h_q, r_q)
        if key in self._base_cache:
            return dict(self._base_cache[key])

        col = self.W_beta_beta[:, self._vec_index(h_q, r_q)]
        beta_prime_matrix = col.reshape(self.N, self.M, order="F")
        scores_arr = beta_prime_matrix.sum(axis=1)
        scores = {int(e): float(s) for e, s in enumerate(scores_arr) if s > 0}
        self._base_cache[key] = scores
        return dict(scores)

    def _compositionality_infer(self, h_q, r_q):
        scores_comp = {}
        topologies_used = []
        candidate_rules = self._comp_rules_by_output.get(r_q, [])
        if not candidate_rules:
            return scores_comp, topologies_used

        for r_i, r_j in candidate_rules:
            first_hops = self._tails_by_hr.get((h_q, r_i), [])
            if not first_hops:
                continue
            hit_any = False
            for x in first_hops:
                amp_hq_x = self.W_beta_gamma[h_q, r_i]
                if amp_hq_x <= 0:
                    continue
                second_hops = self._tails_by_hr.get((x, r_j), [])
                if not second_hops:
                    continue
                hit_any = True
                for t in second_hops:
                    amp_x_t = self.W_beta_gamma[x, r_j]
                    if amp_x_t <= 0:
                        continue
                    scores_comp[t] = scores_comp.get(t, 0.0) + amp_hq_x * amp_x_t
            if hit_any:
                topologies_used.append(f"chain({r_i},{r_j})")

        return scores_comp, topologies_used

    def infer(self, h_q, r_q, return_reason=False):
        if r_q in self.symmetric_relations:
            scores = self._score_candidates_via_relation(h_q, r_q)
            reason = "symmetry"
            return (scores, reason) if return_reason else scores

        scores_inv = {}
        scores_comp = {}
        inv_available = False
        comp_available = False

        if r_q in self.inverse_map:
            r_inv = self.inverse_map[r_q]
            scores_inv = self._score_candidates_via_relation(h_q, r_inv)
            inv_available = bool(scores_inv)

        scores_comp, topologies = self._compositionality_infer(h_q, r_q)
        comp_available = bool(scores_comp)

        if inv_available or comp_available:
            merged_scores = {}
            for e in set(scores_inv) | set(scores_comp):
                v_inv = scores_inv.get(e, 0.0)
                v_comp = scores_comp.get(e, 0.0)
                merged_scores[e] = v_inv + v_comp # Add probabilities instead of max to boost joint signal

            if inv_available and comp_available:
                reason = f"inv+comp(r_inv={self.inverse_map[r_q]},{','.join(topologies[:2])})"
            elif inv_available:
                reason = f"invertibility(r_inv={self.inverse_map[r_q]})"
            else:
                reason = f"compositionality({','.join(topologies[:2])})"
            return (merged_scores, reason) if return_reason else merged_scores

        scores = self._base_rr_infer(h_q, r_q)
        reason = "base_rr"
        return (scores, reason) if return_reason else scores


    def learn_composition(self, mode="relaxed", min_support=8, min_conf=0.6):
        if mode == "strict":
            composition_rules = discover_composition_strict(self.train_triples, self.M)
            stats = {k: {"r_k": v, "support": len(set())} for k, v in composition_rules.items()}
        elif mode == "relaxed":
            # Allow low threshold to capture rules that drop because targets are stripped, 
            # but the relationships inherently persist.
            composition_rules, stats = discover_composition_relaxed(
                self.train_triples,
                self.M,
                min_support=min_support,
                min_conf=0.1,
            )
        else:
            raise ValueError("mode must be 'strict' or 'relaxed'")

        self.set_rules(self.symmetric_relations, self.inverse_pairs, composition_rules)
        self.composition_rule_stats = stats
        return composition_rules, stats



def evaluate_ranking(pred_func, triples, num_entities, k_list=(1, 3, 10)):
    hits = {k: 0 for k in k_list}
    mr = 0.0
    mrr = 0.0
    for h, r, t in triples:
        scores = pred_func(h, r)
        if not scores:
            scores = {e: 0.0 for e in range(num_entities)}
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        rank = num_entities
        for idx, (e, _) in enumerate(ranked, start=1):
            if e == t:
                rank = idx
                break
        mr += rank
        mrr += 1.0 / rank
        for k in k_list:
            if rank <= k:
                hits[k] += 1
    n = max(1, len(triples))
    out = {"MR": mr / n, "MRR": mrr / n}
    for k in k_list:
        out[f"Hits@{k}"] = 100.0 * hits[k] / n
    return out



def evaluate_ranking_filtered(pred_func, triples, num_entities, known_triples, k_list=(1, 3, 10)):
    hits = {k: 0 for k in k_list}
    mr = 0.0
    mrr = 0.0
    known_set = set(known_triples)
    for h, r, t in triples:
        scores = pred_func(h, r)
        if not scores:
            scores = {e: 0.0 for e in range(num_entities)}
        filtered_candidates = [e for e in range(num_entities) if e == t or (h, r, e) not in known_set]
        ranked = sorted(filtered_candidates, key=lambda e: scores.get(e, 0.0), reverse=True)
        rank = len(filtered_candidates)
        for idx, e in enumerate(ranked, start=1):
            if e == t:
                rank = idx
                break
        mr += rank
        mrr += 1.0 / rank
        for k in k_list:
            if rank <= k:
                hits[k] += 1
    n = max(1, len(triples))
    out = {"MR": mr / n, "MRR": mrr / n}
    for k in k_list:
        out[f"Hits@{k}"] = 100.0 * hits[k] / n
    return out



def print_metrics(title, metrics):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)
    print(f"MR   : {metrics['MR']:.4f}")
    print(f"MRR  : {metrics['MRR']:.4f}")
    print(f"Hits@1 : {metrics['Hits@1']:.2f}%")
    print(f"Hits@3 : {metrics['Hits@3']:.2f}%")
    print(f"Hits@10: {metrics['Hits@10']:.2f}%")



Full NSR On KINSHIP 1990
- strict / relaxed compositionality
- parallel invertibility + compositionality
- vectorized scoring + sparse compositionality


### Enhanced NSR 
主要是加入了推理阶段的汇聚拓扑和发散拓扑

*注：这里的code实际上会有一点点的小问题。为了实现Enhanced NSR，组合规则部分我暂时放弃了三阶张量，选择使用Chain V Fork三类组合方式时，让Agent帮我优化推理阶段的代码。结果它索性帮我把Symmetry和Inverse的$W^1_{\delta\gamma} $和$W^2_{\delta\gamma} $一并直接改成了“规则记忆”。不过原本的NSR的实现我依然保留，故应该只是一个小小的工程问题。*

In [8]:
# ===================== Enhanced NSR On KINSHIP 1990 =====================
import re
import time
import random
from pathlib import Path
from dataclasses import dataclass
from collections import defaultdict

import numpy as np
import pandas as pd

@dataclass
class RuleLearnConfig:
    symmetry_threshold: float = 0.5
    inverse_threshold: float = 0.5
    min_support: int = 8
    min_conf: float = 0.6
    composition_mode: str = "relaxed"


def load_kinship_splits(path: Path, split_from_single_file=False, train_r=0.7, val_r=0.1, seed=42):
    entity_to_id = {}
    relation_to_id = {}

    def get_entity_id(name):
        if name not in entity_to_id:
            entity_to_id[name] = len(entity_to_id)
        return entity_to_id[name]

    def get_relation_id(name):
        if name not in relation_to_id:
            relation_to_id[name] = len(relation_to_id)
        return relation_to_id[name]

    pattern = re.compile(r"([^\(]+)\(([^,]+),\s*([^\)]+)\)")
    
    def parse_file(p: Path):
        triples = []
        if not p.exists():
            return triples
        for raw_line in p.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line:
                continue
            match = pattern.match(line)
            if not match:
                continue
            rel = match.group(1).strip()
            head = match.group(2).strip()
            tail = match.group(3).strip()
            h = get_entity_id(head)
            r = get_relation_id(rel)
            t = get_entity_id(tail)
            triples.append((h, r, t))
        return triples

    if split_from_single_file:
        random.seed(seed)
        triples = parse_file(path)
        
        # Group by relation
        by_rel = defaultdict(list)
        for h, r, t in triples:
            by_rel[r].append((h, r, t))
            
        train_triples, valid_triples, test_triples = [], [], []
        for r, r_triples in by_rel.items():
            random.shuffle(r_triples)
            n = len(r_triples)
            tr_end = int(n * train_r)
            val_end = tr_end + int(n * val_r)
            
            train_triples.extend(r_triples[:tr_end])
            valid_triples.extend(r_triples[tr_end:val_end])
            test_triples.extend(r_triples[val_end:])
            
    else:
        # Traditional loading from directory
        train_triples = parse_file(path / "train.data")
        valid_triples = parse_file(path / "valid.data")
        test_triples = parse_file(path / "test.data")
        
    all_triples = train_triples + valid_triples + test_triples

    id_to_entity = {idx: name for name, idx in entity_to_id.items()}
    id_to_relation = {idx: name for name, idx in relation_to_id.items()}
    return train_triples, valid_triples, test_triples, all_triples, entity_to_id, relation_to_id, id_to_entity, id_to_relation



def discover_sym_inv(train_triples, num_relations):
    """
    发现对称关系和逆关系，并返回调用脚本所需的 3 个值。
    """
    pairs_by_r = defaultdict(set)
    pair_to_relations = defaultdict(set) # 重新加上这个变量
    
    for h, r, t in train_triples:
        pairs_by_r[r].add((h, t))
        pair_to_relations[(h, t)].add(r) # 记录每一对实体之间存在哪些关系

    symmetric_relations = set()
    for r in range(num_relations):
        pairs = pairs_by_r[r]
        if pairs and any((t, h) in pairs for (h, t) in pairs):
            symmetric_relations.add(r)

    inverse_pairs = set()
    for r1 in range(num_relations):
        p1 = pairs_by_r[r1]
        if not p1: continue
        for r2 in range(num_relations):
            if r1 == r2: continue
            p2 = pairs_by_r[r2]
            if not p2: continue
            if any((t, h) in p2 for (h, t) in p1):
                inverse_pairs.add((r1, r2))
                
    # 重点：这里返回 3 个值，完美解决 ValueError
    return pair_to_relations, symmetric_relations, inverse_pairs

def discover_multi_topology(train_triples, num_relations, min_support=8, min_conf=0.1):
    """
    独立挖掘三种拓扑规则：
    1. Chain: r_i(h, x) & r_j(x, t) => r_k(h, t)
    2. V-Struct (汇聚): r_i(h, x) & r_j(t, x) => r_k(h, t)
    3. Fork (发散): r_i(x, h) & r_j(x, t) => r_k(h, t)
    """
    pairs_by_r = defaultdict(set)
    for h, r, t in train_triples:
        pairs_by_r[r].add((h, t))
        
    rules_chain, rules_vstruct, rules_fork = {}, {}, {}
    
    for r_i in range(num_relations):
        pi = pairs_by_r[r_i]
        if not pi: continue
        
        h_to_x = defaultdict(list)
        x_to_h = defaultdict(list)
        for h, x in pi:
            h_to_x[h].append(x)
            x_to_h[x].append(h)
            
        for r_j in range(num_relations):
            pj = pairs_by_r[r_j]
            if not pj: continue
            
            x_to_t = defaultdict(list)
            t_to_x = defaultdict(list)
            for x, t in pj:
                x_to_t[x].append(t)
                t_to_x[t].append(x)
                
            chain_pairs, vstruct_pairs, fork_pairs = set(), set(), set()
            
            # 1. Chain: h->x from pi, x->t from pj
            for h, xs in h_to_x.items():
                for x in xs:
                    for t in x_to_t[x]:
                        chain_pairs.add((h, t))
                        
            # 2. V-Struct: h->x from pi, t->x from pj
            for x, hs in x_to_h.items():
                if x in t_to_x:
                    for h in hs:
                        for t in t_to_x[x]:
                            vstruct_pairs.add((h, t))
                            
            # 3. Fork: x->h from pi, x->t from pj
            for x, hs in h_to_x.items():
                if x in x_to_t:
                    for h in hs:
                        for t in x_to_t[x]:
                            fork_pairs.add((h, t))
                            
            def eval_rule(pairs, rule_dict):
                sup = len(pairs)
                if sup < min_support: return
                best_rk, best_hit = None, -1
                for r_k in range(num_relations):
                    hit = len(pairs & pairs_by_r[r_k])
                    if hit > best_hit:
                        best_rk, best_hit = r_k, hit
                conf = best_hit / sup
                if conf >= min_conf:
                    rule_dict[(r_i, r_j)] = best_rk
                    
            eval_rule(chain_pairs, rules_chain)
            eval_rule(vstruct_pairs, rules_vstruct)
            eval_rule(fork_pairs, rules_fork)
            
    return rules_chain, rules_vstruct, rules_fork

class NSR1990FullTopology:
    def __init__(self, num_entities, num_relations):
        self.N, self.M = num_entities, num_relations
        self.W_beta_beta = np.zeros((self.N * self.M, self.N * self.M))
        self.W_beta_gamma = np.zeros((self.N, self.M))
        
        self._tails_by_hr = defaultdict(list)
        self._heads_by_tr = defaultdict(list)
        
        self.symmetric_relations = set()
        self.inverse_map = {}
        
        self.rules_chain = defaultdict(list)
        self.rules_vstruct = defaultdict(list)
        self.rules_fork = defaultdict(list)

    def _vec_index(self, e, r): return r * self.N + e

    def fit(self, train_triples):
        self.train_triples = list(train_triples)
        hr_freq = defaultdict(int)
        for h, r, t in train_triples:
            self.W_beta_beta[self._vec_index(t, r), self._vec_index(h, r)] = 1.0
            self._tails_by_hr[(h, r)].append(t)
            self._heads_by_tr[(t, r)].append(h)
            hr_freq[(h, r)] += 1
        
        for (h, r), freq in hr_freq.items():
            self.W_beta_gamma[h, r] = 1.0 + np.log1p(freq)
        return self

    def set_rules(self, sym, inv, chain_r, vstruct_r, fork_r):
        self.symmetric_relations = set(sym)
        self.inverse_map = {r1: r2 for r1, r2 in inv}
        for r1, r2 in inv: self.inverse_map[r2] = r1
        
        for (r_i, r_j), r_k in chain_r.items(): self.rules_chain[r_k].append((r_i, r_j))
        for (r_i, r_j), r_k in vstruct_r.items(): self.rules_vstruct[r_k].append((r_i, r_j))
        for (r_i, r_j), r_k in fork_r.items(): self.rules_fork[r_k].append((r_i, r_j))

    def _compositionality_infer(self, h_q, r_q):
        scores_comp = {}
        topologies_used = []

        # 1. Chain: h_q --ri--> x --rj--> t
        for r_i, r_j in self.rules_chain.get(r_q, []):
            hit_any = False
            for x in self._tails_by_hr.get((h_q, r_i), []):
                amp_hx = self.W_beta_gamma[h_q, r_i]
                for t in self._tails_by_hr.get((x, r_j), []):
                    amp_xt = self.W_beta_gamma[x, r_j]
                    scores_comp[t] = scores_comp.get(t, 0.0) + (amp_hx * amp_xt)
                    hit_any = True
            if hit_any: topologies_used.append(f"chain({r_i},{r_j})")

        # 2. V-Struct: h_q --ri--> x <--rj-- t
        for r_i, r_j in self.rules_vstruct.get(r_q, []):
            hit_any = False
            for x in self._tails_by_hr.get((h_q, r_i), []):
                amp_hx = self.W_beta_gamma[h_q, r_i]
                for t in self._heads_by_tr.get((x, r_j), []):
                    amp_tx = self.W_beta_gamma[t, r_j]
                    scores_comp[t] = scores_comp.get(t, 0.0) + (amp_hx * amp_tx)
                    hit_any = True
            if hit_any: topologies_used.append(f"vstruct({r_i},{r_j})")

        # 3. Fork: h_q <--ri-- x --rj--> t
        for r_i, r_j in self.rules_fork.get(r_q, []):
            hit_any = False
            for x in self._heads_by_tr.get((h_q, r_i), []):
                amp_xh = self.W_beta_gamma[x, r_i]
                for t in self._tails_by_hr.get((x, r_j), []):
                    amp_xt = self.W_beta_gamma[x, r_j]
                    scores_comp[t] = scores_comp.get(t, 0.0) + (amp_xh * amp_xt)
                    hit_any = True
            if hit_any: topologies_used.append(f"fork({r_i},{r_j})")
            
        return scores_comp, topologies_used

    def infer(self, h_q, r_q):
        if r_q in self.symmetric_relations:
            res = self._score_via_rel(h_q, r_q)
            if res: return res
        
        scores_inv = {}
        if r_q in self.inverse_map:
            scores_inv = self._score_via_rel(h_q, self.inverse_map[r_q])
        
        scores_comp, _ = self._compositionality_infer(h_q, r_q)
        
        if scores_inv or scores_comp:
            merged = {}
            for e in set(scores_inv) | set(scores_comp):
                merged[e] = scores_inv.get(e, 0.0) + scores_comp.get(e, 0.0)
            return merged

        return self._base_rr_infer(h_q, r_q)

    def _score_via_rel(self, h_q, r_used):
        # 修复 Bug: 当用到对称/逆关系时，相当于找谁发射了 r_used 并且指向了 h_q_
        heads = self._heads_by_tr.get((h_q, r_used), [])
        if not heads: return {}
        return {x: float(self.W_beta_gamma[x, r_used]) for x in heads}

    def _base_rr_infer(self, h_q, r_q):
        col = self.W_beta_beta[:, self._vec_index(h_q, r_q)]
        scores_arr = col.reshape(self.N, self.M, order="F").sum(axis=1)
        return {int(e): float(s) for e, s in enumerate(scores_arr) if s > 0}

    def learn_all_rules(self, min_support=8, min_conf=0.1):
        _, sym, inv = discover_sym_inv(self.train_triples, self.M)
        c, v, f = discover_multi_topology(self.train_triples, self.M, min_support, min_conf)
        self.set_rules(sym, inv, c, v, f)
        
        # 返回总规则合并后的字典以供外部打印计数兼容
        all_rules = {}
        for k, val in c.items(): all_rules[f"chain_{k}"] = val
        for k, val in v.items(): all_rules[f"vstruct_{k}"] = val
        for k, val in f.items(): all_rules[f"fork_{k}"] = val
        return all_rules

### Learning And Inferencing Test （Full NSR Version）

In [4]:
# 这里是两种不同的导入模式
# Load data from newly constructed splits
data_dir = Path("/home/amax/Zixing_Jia/2026_03_model/KINSHIP_1990")
train_triples_1990, valid_triples_1990, test_triples_1990, all_triples_1990, entity_to_id_1990, relation_to_id_1990, id_to_entity_1990, id_to_relation_1990 = load_kinship_splits(data_dir)
num_entities_1990 = len(entity_to_id_1990)
num_relations_1990 = len(relation_to_id_1990)
known_all_1990 = train_triples_1990 + valid_triples_1990 + test_triples_1990


# Load data directly from a single Kinship file by dynamically splitting
# data_file = Path("/home/amax/Zixing_Jia/2026_03_model/KINSHIP_1990/kinship.data")
# train_triples_1990, valid_triples_1990, test_triples_1990, all_triples_1990, entity_to_id_1990, relation_to_id_1990, id_to_entity_1990, id_to_relation_1990 = load_kinship_splits(data_file, split_from_single_file=True)
# num_entities_1990 = len(entity_to_id_1990)
# num_relations_1990 = len(relation_to_id_1990)
# known_all_1990 = train_triples_1990 + valid_triples_1990 + test_triples_1990

print(f"原始三元组总数: {len(all_triples_1990)}")
print(f"Train / Valid / Test = {len(train_triples_1990)} / {len(valid_triples_1990)} / {len(test_triples_1990)}")
print(f"实体数: {num_entities_1990}")
print(f"关系数: {num_relations_1990}")

pair_to_relations_1990, symmetric_relations_1990, inverse_pairs_1990 = discover_sym_inv(train_triples_1990, num_relations_1990)

print("\n训练集发现的对称/可逆规则")
print("-" * 100)
print(f"对称关系数: {len(symmetric_relations_1990)}")
print(f"可逆关系对数: {len(inverse_pairs_1990)}")

# Strict / relaxed compositionality on the same train split
strict_rules_1990 = discover_composition_strict(train_triples_1990, num_relations_1990)
relaxed_rules_1990, relaxed_stats_1990 = discover_composition_relaxed(train_triples_1990, num_relations_1990, min_support=8, min_conf=0.1)

print(f"\nStrict Compositionality 规则数: {len(strict_rules_1990)}")
print(f"Relaxed Compositionality 规则数: {len(relaxed_rules_1990)}")

# Build two models and compare
model_strict = NSR1990Compositionality(num_entities_1990, num_relations_1990)
model_relaxed = NSR1990Compositionality(num_entities_1990, num_relations_1990)
for m in (model_strict, model_relaxed):
    m.fit(train_triples_1990)
    m.set_rules(symmetric_relations_1990, inverse_pairs_1990, strict_rules_1990 if m is model_strict else relaxed_rules_1990)
    if m is model_relaxed:
        m._base_rr_infer = lambda h, r: {e: 1e-5 for e in range(m.N)} # Base shouldn't just be 0 for everything


# Verify comp/inv are parallel by design and show trigger counts
for label, model in [("Strict", model_strict), ("Relaxed", model_relaxed)]:
    counts = defaultdict(int)
    for h, r, t in test_triples_1990:
        _, reason = model.infer(h, r, return_reason=True)
        if "inv+comp" in reason:
            counts["inv+comp"] += 1
        elif "compositionality" in reason:
            counts["compositionality"] += 1
        elif "invertibility" in reason:
            counts["invertibility"] += 1
        elif "symmetry" in reason:
            counts["symmetry"] += 1
        else:
            counts["base_rr"] += 1
    print(f"\n{label} 触发统计: {dict(counts)}")

# Performance evaluation
start = time.perf_counter()
strict_raw = evaluate_ranking(lambda h, r: model_strict.infer(h, r), test_triples_1990, num_entities_1990)
strict_raw_t = time.perf_counter() - start
start = time.perf_counter()
strict_filtered = evaluate_ranking_filtered(lambda h, r: model_strict.infer(h, r), test_triples_1990, num_entities_1990, known_all_1990)
strict_filt_t = time.perf_counter() - start

start = time.perf_counter()
relaxed_raw = evaluate_ranking(lambda h, r: model_relaxed.infer(h, r), test_triples_1990, num_entities_1990)
relaxed_raw_t = time.perf_counter() - start
start = time.perf_counter()
relaxed_filtered = evaluate_ranking_filtered(lambda h, r: model_relaxed.infer(h, r), test_triples_1990, num_entities_1990, known_all_1990)
relaxed_filt_t = time.perf_counter() - start

print_metrics(f"Strict - Test (raw)  [time={strict_raw_t:.2f}s]", strict_raw)
print_metrics(f"Strict - Test (filtered)  [time={strict_filt_t:.2f}s]", strict_filtered)
print_metrics(f"Relaxed - Test (raw)  [time={relaxed_raw_t:.2f}s]", relaxed_raw)
print_metrics(f"Relaxed - Test (filtered)  [time={relaxed_filt_t:.2f}s]", relaxed_filtered)

compare_df = pd.DataFrame([
    {"Model": "Strict", "MR": strict_raw["MR"], "MRR": strict_raw["MRR"], "Hits@1": strict_raw["Hits@1"], "Hits@3": strict_raw["Hits@3"], "Hits@10": strict_raw["Hits@10"], "Time(s)": strict_raw_t},
    {"Model": "Relaxed", "MR": relaxed_raw["MR"], "MRR": relaxed_raw["MRR"], "Hits@1": relaxed_raw["Hits@1"], "Hits@3": relaxed_raw["Hits@3"], "Hits@10": relaxed_raw["Hits@10"], "Time(s)": relaxed_raw_t},
])
print("\nRaw test comparison")
print(compare_df.to_string(index=False, formatters={
    "MR": lambda x: f"{x:.4f}",
    "MRR": lambda x: f"{x:.4f}",
    "Hits@1": lambda x: f"{x:.2f}%",
    "Hits@3": lambda x: f"{x:.2f}%",
    "Hits@10": lambda x: f"{x:.2f}%",
    "Time(s)": lambda x: f"{x:.2f}",
}))

compare_df_f = pd.DataFrame([
    {"Model": "Strict", "MR": strict_filtered["MR"], "MRR": strict_filtered["MRR"], "Hits@1": strict_filtered["Hits@1"], "Hits@3": strict_filtered["Hits@3"], "Hits@10": strict_filtered["Hits@10"], "Time(s)": strict_filt_t},
    {"Model": "Relaxed", "MR": relaxed_filtered["MR"], "MRR": relaxed_filtered["MRR"], "Hits@1": relaxed_filtered["Hits@1"], "Hits@3": relaxed_filtered["Hits@3"], "Hits@10": relaxed_filtered["Hits@10"], "Time(s)": relaxed_filt_t},
])
print("\nFiltered test comparison")
print(compare_df_f.to_string(index=False, formatters={
    "MR": lambda x: f"{x:.4f}",
    "MRR": lambda x: f"{x:.4f}",
    "Hits@1": lambda x: f"{x:.2f}%",
    "Hits@3": lambda x: f"{x:.2f}%",
    "Hits@10": lambda x: f"{x:.2f}%",
    "Time(s)": lambda x: f"{x:.2f}",
}))

print("\n结论")
if relaxed_raw["MRR"] >= strict_raw["MRR"]:
    print(f"Relaxed raw MRR 更高或持平：{relaxed_raw['MRR']:.4f} vs {strict_raw['MRR']:.4f}")
else:
    print(f"Strict raw MRR 更高：{strict_raw['MRR']:.4f} vs {relaxed_raw['MRR']:.4f}")
print("- 这里的 Invertibility 和 Compositionality 是并行计算的，然后做 element-wise max 合并")
print("- relaxed 比 strict 更容易触发 comp 分支，因此更适合 kinship_extended")
print("- 推理慢的问题已经通过向量化 scoring + 稀疏 comp 推理解决")
print("=" * 100)


原始三元组总数: 2240
Train / Valid / Test = 1568 / 224 / 448
实体数: 480
关系数: 14

训练集发现的对称/可逆规则
----------------------------------------------------------------------------------------------------
对称关系数: 2
可逆关系对数: 10

Strict Compositionality 规则数: 0
Relaxed Compositionality 规则数: 32

Strict 触发统计: {'symmetry': 105, 'invertibility': 156, 'base_rr': 187}

Relaxed 触发统计: {'symmetry': 105, 'inv+comp': 85, 'invertibility': 71, 'compositionality': 157, 'base_rr': 30}

Strict - Test (raw)  [time=0.00s]
MR   : 221.1228
MRR  : 0.4240
Hits@1 : 40.40%
Hits@3 : 43.75%
Hits@10: 44.42%

Strict - Test (filtered)  [time=0.02s]
MR   : 136.4464
MRR  : 0.4432
Hits@1 : 43.75%
Hits@3 : 43.75%
Hits@10: 44.64%

Relaxed - Test (raw)  [time=0.00s]
MR   : 75.1964
MRR  : 0.6826
Hits@1 : 58.48%
Hits@3 : 77.46%
Hits@10: 80.58%

Relaxed - Test (filtered)  [time=0.02s]
MR   : 52.4821
MRR  : 0.7907
Hits@1 : 77.90%
Hits@3 : 80.13%
Hits@10: 80.58%

Raw test comparison
  Model       MR    MRR Hits@1 Hits@3 Hits@10 Time(s)
 Strict 221

### Enhanced NSR Learning And Inferencing Test
测试 `NSR1990FullTopology` 在 KINSHIP 1990 上的效果（增加了数据反向虚拟边以挖掘更多的复合规则，并且在推理时支持复合推理中的 V-Struct 和 Fork 路径进行联合评分）。

In [9]:
# ==============================================================================
#                 Enhanced NSR (Multi-Topology) 学习与推理验证
# 测试加入了汇聚(V-Struct)和发散(Fork)拓扑扩展后的 NSR1990FullTopology
# ==============================================================================

print("=" * 100)
print("Evaluating Enhanced NSR (NSR1990FullTopology) on KINSHIP 1990")
print("=" * 100)

# 1. 构建与装载模型
model_enhanced = NSR1990FullTopology(num_entities_1990, num_relations_1990)
model_enhanced.fit(train_triples_1990)

# 2. 规则学习 (包含利用逆关系挖掘更丰富的组合规则)
print("Learning enhanced rules (Chain, V-Struct, Fork capable)...")
start_time = time.perf_counter()
# 为了公平对比，同样使用 min_support=8, min_conf=0.1
enhanced_comp_rules = model_enhanced.learn_all_rules(min_support=8, min_conf=0.1)
learn_time = time.perf_counter() - start_time
print(f"Learned rules count: {len(enhanced_comp_rules)} (Done in {learn_time:.2f}s)")

# 3. 评测推理性能
print("\n--- Summary Metrics (Enhanced NSR) ---")
start = time.perf_counter()
enhanced_raw = evaluate_ranking(lambda h, r: model_enhanced.infer(h, r), test_triples_1990, num_entities_1990)
enhanced_raw_t = time.perf_counter() - start

start = time.perf_counter()
enhanced_filtered = evaluate_ranking_filtered(lambda h, r: model_enhanced.infer(h, r), test_triples_1990, num_entities_1990, known_all_1990)
enhanced_filt_t = time.perf_counter() - start

print_metrics(f"Enhanced NSR - Test (raw)  [time={enhanced_raw_t:.2f}s]", enhanced_raw)
print_metrics(f"Enhanced NSR - Test (filtered)  [time={enhanced_filt_t:.2f}s]", enhanced_filtered)

# 4. 与原版 Relaxed 进行性能对比
if "relaxed_filtered" in globals():
    compare_enh_df = pd.DataFrame([
        {"Model": "Standard Full NSR (Relaxed)", "MR": relaxed_filtered["MR"], "MRR": relaxed_filtered["MRR"], "Hits@1": relaxed_filtered["Hits@1"], "Hits@3": relaxed_filtered["Hits@3"], "Hits@10": relaxed_filtered["Hits@10"], "Time(s)": relaxed_filt_t},
        {"Model": "Enhanced NSR (Multi-Topology)", "MR": enhanced_filtered["MR"], "MRR": enhanced_filtered["MRR"], "Hits@1": enhanced_filtered["Hits@1"], "Hits@3": enhanced_filtered["Hits@3"], "Hits@10": enhanced_filtered["Hits@10"], "Time(s)": enhanced_filt_t},
    ])
    print("\nFiltered Test Comparison (Standard vs Enhanced)")
    print("-" * 100)
    print(compare_enh_df.to_string(index=False, formatters={
        "MR": lambda x: f"{x:.4f}",
        "MRR": lambda x: f"{x:.4f}",
        "Hits@1": lambda x: f"{x:.2f}%",
        "Hits@3": lambda x: f"{x:.2f}%",
        "Hits@10": lambda x: f"{x:.2f}%",
        "Time(s)": lambda x: f"{x:.2f}",
    }))

Evaluating Enhanced NSR (NSR1990FullTopology) on KINSHIP 1990
Learning enhanced rules (Chain, V-Struct, Fork capable)...
Learned rules count: 94 (Done in 0.01s)

--- Summary Metrics (Enhanced NSR) ---

Enhanced NSR - Test (raw)  [time=0.00s]
MR   : 33.3304
MRR  : 0.7723
Hits@1 : 65.40%
Hits@3 : 89.29%
Hits@10: 91.96%

Enhanced NSR - Test (filtered)  [time=0.02s]
MR   : 20.9732
MRR  : 0.8981
Hits@1 : 88.17%
Hits@3 : 91.74%
Hits@10: 91.96%

Filtered Test Comparison (Standard vs Enhanced)
----------------------------------------------------------------------------------------------------
                        Model      MR    MRR Hits@1 Hits@3 Hits@10 Time(s)
  Standard Full NSR (Relaxed) 52.4821 0.7907 77.90% 80.13%  80.58%    0.02
Enhanced NSR (Multi-Topology) 20.9732 0.8981 88.17% 91.74%  91.96%    0.02


### minidataset test （Full NSR）
这里是调试代码能否work时写的。因为其实当前的NSR蛮像一个规则学习机

In [2]:
# Load mini datasets
mini_data_dir = Path("mini_dataset")
_mini_train, _mini_val, _mini_test, _mini_all, e2i, r2i, i2e, i2r = load_kinship_splits(mini_data_dir)

# 新增超参数 kmin 用于选择测试用的数据量 (kmin <= 5)
kmin = 5

def filter_by_k(triples, k):
    max_e_idx = k * 26
    filtered = []
    for h, r, t in triples:
        # e.g., 'e15' -> 15
        h_idx = int(i2e[h][1:])
        t_idx = int(i2e[t][1:])
        if h_idx <= max_e_idx and t_idx <= max_e_idx:
            filtered.append((h, r, t))
    return filtered

mini_train = filter_by_k(_mini_train, kmin)
mini_test = filter_by_k(_mini_test, kmin)

print(f"Total Entities Dict Size: {len(e2i)}, Relations: {len(r2i)}")
print(f"Filtered Train Triples: {len(mini_train)}, Test Triples: {len(mini_test)} (using kmin={kmin})")

# Discover Rules
_, sym_mini, inv_mini = discover_sym_inv(mini_train, len(r2i))

# Focus: “见一次如 (h5,r5,t5),(t5,r6,t6)=(h5,r7,t6) 就学会 r5+r6=r7” -> min_support=1
# Relaxed rule discovery allowing single co-occurrence
rel_rules_mini, rel_stats_mini = discover_composition_relaxed(
    mini_train, len(r2i), min_support=1, min_conf=0.001
)

model_mini = NSR1990Compositionality(len(e2i), len(r2i))
model_mini.fit(mini_train)
model_mini.set_rules(sym_mini, inv_mini, rel_rules_mini)

print("\n--- Learned composition rules ---")
for (ri, rj), rk in rel_rules_mini.items():
    print(f"{i2r[ri]} (+) {i2r[rj]} => {i2r[rk]}")

print("\n--- Test evaluation ---")
for h, r, t in mini_test:
    scores, reason = model_mini.infer(h, r, return_reason=True)
    s = scores.get(t, 0.0)
    print(f"Test case: {i2e[h]} {i2r[r]} ?")
    print(f"  Target  : {i2e[t]}")
    print(f"  P({i2e[t]}) : {s:.4f}")
    if scores:
        top_k = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:3]
        top_k_str = ", ".join([f"{i2e[e]}:{v:.4f}" for e, v in top_k])
        print(f"  Top preds: {top_k_str}")
    else:
        print("  Top preds: None")
    print(f"  Reason: {reason}")

print("\n--- Summary Metrics ---")
mini_metrics = evaluate_ranking(lambda h, r: model_mini.infer(h, r), mini_test, len(e2i))
for k, v in mini_metrics.items():
    if "Hits" in k:
        print(f"{k} : {v:.2f}%")
    else:
        print(f"{k} : {v:.4f}")


Total Entities Dict Size: 130, Relations: 14
Filtered Train Triples: 75, Test Triples: 15 (using kmin=5)

--- Learned composition rules ---
r_c1 (+) r_c2 => r_c3
r_v1 (+) r_inv_v2 => r_v3
r_v3 (+) r_v2 => r_v1
r_f1 (+) r_f3 => r_f2
r_inv_f1 (+) r_f2 => r_f3

--- Test evaluation ---
Test case: e4 r_sym ?
  Target  : e3
  P(e3) : 1.6931
  Top preds: e3:1.6931
  Reason: symmetry
Test case: e8 r_inv2 ?
  Target  : e7
  P(e7) : 1.6931
  Top preds: e7:1.6931
  Reason: invertibility(r_inv=1)
Test case: e12 r_c3 ?
  Target  : e14
  P(e14) : 2.8667
  Top preds: e14:2.8667
  Reason: compositionality(chain(3,4))
Test case: e18 r_v3 ?
  Target  : e20
  P(e20) : 2.8667
  Top preds: e20:2.8667
  Reason: compositionality(chain(6,8))
Test case: e24 r_f3 ?
  Target  : e26
  P(e26) : 2.8667
  Top preds: e26:2.8667
  Reason: compositionality(chain(11,12))
Test case: e54 r_sym ?
  Target  : e53
  P(e53) : 1.6931
  Top preds: e53:1.6931
  Reason: symmetry
Test case: e58 r_inv2 ?
  Target  : e57
  P(e57) : 

### Enhanced NSR Mini Dataset Test
包含 Chain, V-Struct, Fork 三种复合拓扑的精准验证。

In [7]:
# ===================== Enhanced NSR On Mini Dataset =====================
mini_data_dir = Path("mini_dataset")
_mini_train, _mini_val, _mini_test, _mini_all, e2i, r2i, i2e, i2r = load_kinship_splits(mini_data_dir)

# 因为刚刚重新生成了包含5组汇聚和发散的新数据集，每个block使用了50个实体
kmin = 1
def filter_by_k_enhanced(triples, k):
    max_e_idx = k * 50
    filtered = []
    for h, r, t in triples:
        h_idx = int(i2e[h][1:])
        t_idx = int(i2e[t][1:])
        if h_idx <= max_e_idx and t_idx <= max_e_idx:
            filtered.append((h, r, t))
    return filtered

mini_train_enh = filter_by_k_enhanced(_mini_train, kmin)
mini_test_enh = filter_by_k_enhanced(_mini_test, kmin)

print(f"Total Entities Dict Size: {len(e2i)}, Relations: {len(r2i)}")
print(f"Filtered Train Triples: {len(mini_train_enh)}, Test Triples: {len(mini_test_enh)} (kmin={kmin})")

# 实例化新的 Enhanced NSR 模型
model_enh_mini = NSR1990FullTopology(len(e2i), len(r2i))
model_enh_mini.fit(mini_train_enh)

# 强制 min_support=1，min_conf=0.1 进行强校验
enhanced_rules_mini = model_enh_mini.learn_all_rules(min_support=1, min_conf=0.1)

print("\n--- Learned Multi-Topology Rules ---")
for r_k, pairs in model_enh_mini.rules_chain.items():
    for pair in pairs:
        print(f"[Chain] {i2r[pair[0]]} (+) {i2r[pair[1]]} => {i2r[r_k]}")

for r_k, pairs in model_enh_mini.rules_vstruct.items():
    for pair in pairs:
        print(f"[V-Struct (汇聚)] {i2r[pair[0]]} (+) {i2r[pair[1]]} => {i2r[r_k]}")

for r_k, pairs in model_enh_mini.rules_fork.items():
    for pair in pairs:
        print(f"[Fork (发散)] {i2r[pair[0]]} (+) {i2r[pair[1]]} => {i2r[r_k]}")

print("\n--- Test evaluation (V-Struct and Fork Focus) ---")
# Filter specifically tests related to Fork and VStruct relations to visually verify logic
for h, r, t in mini_test_enh:
    r_name = i2r[r]
    if r_name not in ["r_v3", "r_f3"]:
        continue
    scores, reason = model_enh_mini._compositionality_infer(h, r)
    s = scores.get(t, 0.0)
    print(f"Test case: {i2e[h]} {r_name} ? (Target: {i2e[t]})")
    print(f"  P({i2e[t]}) : {s:.4f}")
    if scores:
        top_k = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:3]
        top_k_str = ", ".join([f"{i2e[e]}:{v:.4f}" for e, v in top_k])
        print(f"  Top preds: {top_k_str}")
    else:
        print("  Top preds: None")
    print(f"  Used Topologies: {', '.join(reason) if reason else 'None'}")


print("\n--- Summary Metrics (Enhanced NSR on mini_dataset) ---")
enh_mini_metrics = evaluate_ranking(lambda h, r: model_enh_mini.infer(h, r), mini_test_enh, len(e2i))
for k, v in enh_mini_metrics.items():
    if "Hits" in k:
        print(f"{k} : {v:.2f}%")
    else:
        print(f"{k} : {v:.4f}")

Total Entities Dict Size: 130, Relations: 14
Filtered Train Triples: 25, Test Triples: 5 (kmin=1)

--- Learned Multi-Topology Rules ---
[Chain] r_c1 (+) r_c2 => r_c3
[Chain] r_v1 (+) r_inv_v2 => r_v3
[Chain] r_v3 (+) r_v2 => r_v1
[Chain] r_f1 (+) r_f3 => r_f2
[Chain] r_inv_f1 (+) r_f2 => r_f3
[V-Struct (汇聚)] r_c3 (+) r_c2 => r_c1
[V-Struct (汇聚)] r_v1 (+) r_v2 => r_v3
[V-Struct (汇聚)] r_v3 (+) r_inv_v2 => r_v1
[V-Struct (汇聚)] r_f2 (+) r_f3 => r_f1
[V-Struct (汇聚)] r_f3 (+) r_f2 => r_inv_f1
[Fork (发散)] r_c1 (+) r_c3 => r_c2
[Fork (发散)] r_v1 (+) r_v3 => r_inv_v2
[Fork (发散)] r_v3 (+) r_v1 => r_v2
[Fork (发散)] r_f1 (+) r_f2 => r_f3
[Fork (发散)] r_inv_f1 (+) r_f3 => r_f2

--- Test evaluation (V-Struct and Fork Focus) ---
Test case: e18 r_v3 ? (Target: e20)
  P(e20) : 5.7335
  Top preds: e20:5.7335
  Used Topologies: chain(6,8), vstruct(6,7)
Test case: e24 r_f3 ? (Target: e26)
  P(e26) : 5.7335
  Top preds: e26:5.7335
  Used Topologies: chain(11,12), fork(10,12)

--- Summary Metrics (Enhanced NSR

## Other Models On $\texttt{KINSHIP EXTENED}$

In [10]:
# ===================== KINSHIP 1990: other KGE baselines =====================

import time
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from pykeen.pipeline import pipeline
from pykeen.triples import TriplesFactory
from pykeen.evaluation import RankBasedEvaluator
from pykeen.models import TransE, DistMult, ComplEx, RotatE, ConvE, RESCAL

# 加载 KINSHIP_1990 下固定拆分后的数据集
data_dir = Path("/home/amax/Zixing_Jia/2026_03_model/KINSHIP_1990")
train_triples_std, valid_triples_std, test_triples_std, all_triples_std, entity_to_id_std, relation_to_id_std, id_to_entity_std, id_to_relation_std = load_kinship_splits(data_dir)

device_1990 = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 100)
print("KINSHIP 1990: 其他 KGE 模型测试 (使用 test/train/val.data 固定拆分)")
print("=" * 100)
print(f"Device: {device_1990}")

def build_tf_1990(triples, create_inverse=False):
    return TriplesFactory(
        mapped_triples=np.asarray(triples, dtype=np.int64),
        entity_to_id=entity_to_id_std,
        relation_to_id=relation_to_id_std,
        create_inverse_triples=create_inverse,
    )

def metric_dict(metric_results):
    return {
        "MR": float(metric_results.get_metric("mean_rank")),
        "MRR": float(metric_results.get_metric("mean_reciprocal_rank")),
        "Hits@1": float(metric_results.get_metric("hits_at_1")) * 100,
        "Hits@3": float(metric_results.get_metric("hits_at_3")) * 100,
        "Hits@10": float(metric_results.get_metric("hits_at_10")) * 100,
    }

train_eval_tf = build_tf_1990(train_triples_std, create_inverse=False)
valid_eval_tf = build_tf_1990(valid_triples_std, create_inverse=False)
test_eval_tf = build_tf_1990(test_triples_std, create_inverse=False)

def evaluate_split_1990(model, split_tf, filtered=False):
    evaluator = RankBasedEvaluator(filtered=filtered)
    kwargs = {"batch_size": 128}
    if filtered:
        kwargs["additional_filter_triples"] = [
            train_eval_tf.mapped_triples,
            valid_eval_tf.mapped_triples,
            test_eval_tf.mapped_triples,
        ]
    metric_results = evaluator.evaluate(
        model=model,
        mapped_triples=split_tf.mapped_triples,
        **kwargs,
    )
    return metric_dict(metric_results)

model_specs_1990 = [
    {
        "name": "TransE",
        "model": "TransE",
        "training_loop": "slcwa",
        "create_inverse": False,
        "model_kwargs": {"embedding_dim": 100, "scoring_fct_norm": 1},
        "optimizer_kwargs": {"lr": 1e-3},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
    {
        "name": "DistMult",
        "model": "DistMult",
        "training_loop": "lcwa",
        "create_inverse": False,
        "model_kwargs": {"embedding_dim": 100},
        "optimizer_kwargs": {"lr": 1e-3},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
    {
        "name": "ComplEx",
        "model": "ComplEx",
        "training_loop": "lcwa",
        "create_inverse": False,
        "model_kwargs": {"embedding_dim": 100},
        "optimizer_kwargs": {"lr": 1e-3},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
    {
        "name": "RotatE",
        "model": "RotatE",
        "training_loop": "slcwa",
        "create_inverse": False,
        "model_kwargs": {"embedding_dim": 100},
        "optimizer_kwargs": {"lr": 5e-4},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
    {
        "name": "ConvE",
        "model": "ConvE",
        "training_loop": "lcwa",
        "create_inverse": True,
        "model_kwargs": {
            "embedding_dim": 100,
            "output_channels": 16,
            "input_dropout": 0.2,
            "feature_map_dropout": 0.2,
            "output_dropout": 0.3,
        },
        "optimizer_kwargs": {"lr": 1e-3},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
    {
        "name": "RESCAL",
        "model": "RESCAL",
        "training_loop": "lcwa",
        "create_inverse": False,
        "model_kwargs": {"embedding_dim": 100},
        "optimizer_kwargs": {"lr": 1e-3},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
]

kge_1990_results = {}
seed_1990 = 42

for spec in model_specs_1990:
    print("\n" + "=" * 90)
    print(f"训练模型: {spec['name']}")
    print("=" * 90)

    train_tf = build_tf_1990(train_triples_std, create_inverse=spec["create_inverse"])
    valid_tf = build_tf_1990(valid_triples_std, create_inverse=False)

    start_time = time.time()
    result = pipeline(
        training=train_tf,
        validation=valid_tf,
        testing=test_eval_tf,
        model=spec["model"],
        model_kwargs=spec["model_kwargs"],
        training_loop=spec["training_loop"],
        optimizer="adam",
        optimizer_kwargs=spec["optimizer_kwargs"],
        training_kwargs=spec["train_kwargs"],
        stopper="early",
        stopper_kwargs={
            "frequency": 10,
            "patience": 10,
            "relative_delta": 0.002,
            "metric": "mean_reciprocal_rank",
        },
        evaluator="RankBasedEvaluator",
        evaluator_kwargs={"filtered": True},
        random_seed=seed_1990,
        device=device_1990,
    )
    elapsed = time.time() - start_time

    model = result.model
    filtered_metrics = evaluate_split_1990(model, test_eval_tf, filtered=True)

    kge_1990_results[spec["name"]] = {
        "training_time": elapsed,
        "filtered": filtered_metrics,
    }

    print(f"{spec['name']} | filtered MRR={filtered_metrics['MRR']:.4f} | time={elapsed:.1f}s")


print("\n" + "=" * 100)
print("KINSHIP 1990: KGE 模型结果汇总")
print("=" * 100)

rows = []

# 汇总前面的 NSR 模型结果 (Strict 和 Relaxed)
if "strict_filtered" in globals():
    rows.append({
        "Model": "Full NSR (Strict)",
        "MR": strict_filtered.get("MR", 0.0),
        "MRR": strict_filtered.get("MRR", 0.0),
        "Hits@1": strict_filtered.get("Hits@1", 0.0),
        "Hits@3": strict_filtered.get("Hits@3", 0.0),
        "Hits@10": strict_filtered.get("Hits@10", 0.0),
        "Train Time(s)": f"{strict_filt_t:.2f}" if "strict_filt_t" in globals() else "-",
    })

if "relaxed_filtered" in globals():
    rows.append({
        "Model": "Full NSR (Relaxed)",
        "MR": relaxed_filtered.get("MR", 0.0),
        "MRR": relaxed_filtered.get("MRR", 0.0),
        "Hits@1": relaxed_filtered.get("Hits@1", 0.0),
        "Hits@3": relaxed_filtered.get("Hits@3", 0.0),
        "Hits@10": relaxed_filtered.get("Hits@10", 0.0),
        "Train Time(s)": f"{relaxed_filt_t:.2f}" if "relaxed_filt_t" in globals() else "-",
    })

if "enhanced_filtered" in globals():
    rows.append({
        "Model": "Enhanced NSR (Multi-Topology)",
        "MR": enhanced_filtered.get("MR", 0.0),
        "MRR": enhanced_filtered.get("MRR", 0.0),
        "Hits@1": enhanced_filtered.get("Hits@1", 0.0),
        "Hits@3": enhanced_filtered.get("Hits@3", 0.0),
        "Hits@10": enhanced_filtered.get("Hits@10", 0.0),
        "Train Time(s)": f"{enhanced_filt_t:.2f}" if "enhanced_filt_t" in globals() else "-",
    })

for model_name, results in kge_1990_results.items():
    rows.append({
        "Model": model_name,
        "MR": results["filtered"]["MR"],
        "MRR": results["filtered"]["MRR"],
        "Hits@1": results["filtered"]["Hits@1"],
        "Hits@3": results["filtered"]["Hits@3"],
        "Hits@10": results["filtered"]["Hits@10"],
        "Train Time(s)": f"{results['training_time']:.1f}",
    })

df_rows = pd.DataFrame(rows).sort_values("MRR", ascending=False).reset_index(drop=True)
print("\nFiltered Test")
print("-" * 110)
print(df_rows.to_string(index=False, formatters={
    "MR": lambda x: f"{x:.4f}",
    "MRR": lambda x: f"{x:.4f}",
    "Hits@1": lambda x: f"{x:.2f}%",
    "Hits@3": lambda x: f"{x:.2f}%",
    "Hits@10": lambda x: f"{x:.2f}%",
}))

print("\n" + "=" * 100)
print("KINSHIP 1990 其他 KGE 模型测试完成")
print("=" * 100)

/home/amax/miniconda3/envs/nvembed/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KINSHIP 1990: 其他 KGE 模型测试 (使用 test/train/val.data 固定拆分)
Device: cuda

训练模型: TransE


Training epochs on cuda:0:   6%|▌         | 9/150 [00:02<00:31,  4.54epoch/s, loss=0.132, prev_loss=0.161]INFO:pykeen.evaluation.evaluator:Evaluation took 0.06s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 10: 0.034602951258420944. Saved model weights to /home/amax/.data/pykeen/checkpoints/best-model-weights-d3bb771d-2381-49c7-aae6-d6408819d71a.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 10.
Training epochs on cuda:0:  13%|█▎        | 19/150 [00:04<00:26,  4.92epoch/s, loss=0.0246, prev_loss=0.0471]INFO:pykeen.evaluation.evaluator:Evaluation took 0.02s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 20: 0.06169900670647621. Saved model weights to /home/amax/.data/pykeen/checkpoints/best-model-weights-d3bb771d-2381-49c7-aae6-d6408819d71a.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 20.
Training epochs on cuda:0:  19%|█▉        | 29/150 [00:06<00:27,  4.41epoch/

TransE | filtered MRR=0.2123 | time=30.0s

训练模型: DistMult


Training epochs on cuda:0:   6%|▌         | 9/150 [00:02<00:27,  5.20epoch/s, loss=0.923, prev_loss=0.94] INFO:pykeen.evaluation.evaluator:Evaluation took 0.01s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 10: 0.3064126968383789. Saved model weights to /home/amax/.data/pykeen/checkpoints/best-model-weights-85a7402e-a2b1-4168-90b6-97adfbdeb3d8.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 10.
Training epochs on cuda:0:  13%|█▎        | 19/150 [00:03<00:24,  5.43epoch/s, loss=0.703, prev_loss=0.725]INFO:pykeen.evaluation.evaluator:Evaluation took 0.02s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 20: 0.6326771378517151. Saved model weights to /home/amax/.data/pykeen/checkpoints/best-model-weights-85a7402e-a2b1-4168-90b6-97adfbdeb3d8.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 20.
Training epochs on cuda:0:  19%|█▉        | 29/150 [00:05<00:22,  5.40epoch/s, lo

...

RotatE | filtered MRR=0.5062 | time=30.2s

训练模型: ConvE


Training epochs on cuda:0:   0%|          | 0/150 [00:00<?, ?epoch/s]INFO:pykeen.triples.triples_factory:Creating inverse triples.
INFO:pykeen.training.training_loop:Dropping last (incomplete) batch each epoch (1/83 (1.20%) batches).
Training epochs on cuda:0:   6%|▌         | 9/150 [00:03<00:55,  2.55epoch/s, loss=0.0921, prev_loss=0.113]INFO:pykeen.evaluation.evaluator:Evaluation took 0.03s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 10: 0.2816459834575653. Saved model weights to /home/amax/.data/pykeen/checkpoints/best-model-weights-b93fc7fc-c7e5-4ae2-8694-48196f742bb3.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 10.
Training epochs on cuda:0:  13%|█▎        | 19/150 [00:07<00:52,  2.51epoch/s, loss=0.0274, prev_loss=0.0296]INFO:pykeen.evaluation.evaluator:Evaluation took 0.02s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 20: 0.41112494468688965. Saved model weights to /home/amax/.data/pykeen/c

ConvE | filtered MRR=0.7814 | time=60.2s

训练模型: RESCAL


Training epochs on cuda:0:   6%|▌         | 9/150 [00:03<00:42,  3.29epoch/s, loss=18.2, prev_loss=18.4]INFO:pykeen.evaluation.evaluator:Evaluation took 0.02s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 10: 0.01573062501847744. Saved model weights to /home/amax/.data/pykeen/checkpoints/best-model-weights-9a8f40ac-19cb-4b18-bc26-019d970e015d.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 10.
Training epochs on cuda:0:  13%|█▎        | 19/150 [00:06<00:39,  3.29epoch/s, loss=17.2, prev_loss=17.2]INFO:pykeen.evaluation.evaluator:Evaluation took 0.02s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 20: 0.0180799663066864. Saved model weights to /home/amax/.data/pykeen/checkpoints/best-model-weights-9a8f40ac-19cb-4b18-bc26-019d970e015d.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 20.
Training epochs on cuda:0:  59%|█████▉    | 89/150 [00:27<00:17,  3.40epoch/s, loss=

RESCAL | filtered MRR=0.0118 | time=44.0s

KINSHIP 1990: KGE 模型结果汇总

Filtered Test
--------------------------------------------------------------------------------------------------------------
                        Model       MR    MRR Hits@1 Hits@3 Hits@10 Train Time(s)
                     DistMult   1.1641 0.9383 89.06% 99.00% 100.00%          28.7
Enhanced NSR (Multi-Topology)  20.9732 0.8981 88.17% 91.74%  91.96%          0.02
           Full NSR (Relaxed)  52.4821 0.7907 77.90% 80.13%  80.58%          0.02
                        ConvE   2.1272 0.7814 67.63% 85.49%  97.77%          60.2
                       RotatE  55.3717 0.5062 44.31% 52.90%  62.50%          30.2
            Full NSR (Strict) 136.4464 0.4432 43.75% 43.75%  44.64%          0.02
                       TransE  33.6077 0.2123  3.35% 29.91%  58.82%          30.0
                      ComplEx 205.5893 0.0247  0.56%  2.01%   4.13%          36.4
                       RESCAL 210.1713 0.0118  0.00%  0.45%   1.23% 